# Grid Up Datathon — 01 · Veri Keşfi

**Takım:** _(takım adını yaz)_ · **Tarih:** _(gün)_

Bu notebook, veri setinin ilk saatinde çalıştırılır. Amacı üç soruyu cevaplamak:

1. **Elimizde ne var?** — kolonlar, tipler, eksikler, boyut
2. **Hangi doğrulama şeması doğru?** — zaman var mı, tekrarlayan varlık var mı
3. **Sızıntı var mı?** — modeli eğitmeden önce bilmemiz gereken tek şey

> Bu üç çıktı, sonraki 12 günün her kararını belirler.

In [ ]:
import sys
from pathlib import Path

# Kaggle'da: /kaggle/input/<yarisma>/  · yerelde: data/raw/
IS_KAGGLE = Path("/kaggle/input").exists()
if not IS_KAGGLE:
    sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gridup import environment_report, profile, read_any, set_global_seed
from gridup.compat import categorical_columns
from gridup.profiling import quick_look
from gridup.turkish import codepoints, has_combining_dot, join_key
from gridup.validation import leakage_report, suggest_scheme

set_global_seed(42)

# Ortamı yazdır — jüri tekrarlanabilirliğe bakıyor, bu ucuz bir puan.
for key, value in environment_report().items():
    print(f"{key:<26} {value}")

## 1 · Veriyi oku

`read_any` kodlamayı, ayırıcıyı ve ondalık işaretini **otomatik tespit eder**.
Türk kurumlarından gelen dosyalar `cp1254` + `;` + ondalık `,` olabilir; düz
`pd.read_csv` bunları sessizce bozar.

In [ ]:
DATA_DIR = Path("/kaggle/input/GRID-UP-YARISMA-SLUG") if IS_KAGGLE else Path("../data/raw")

train = read_any(DATA_DIR / "train.csv")
test  = read_any(DATA_DIR / "test.csv")

try:
    sample_submission = read_any(DATA_DIR / "sample_submission.csv")
    print("sample_submission kolonları:", list(sample_submission.columns))
except FileNotFoundError:
    sample_submission = None
    print("sample_submission bulunamadı — dosya adlarını kontrol et:")
    print(sorted(p.name for p in DATA_DIR.glob("*")))

print(f"\ntrain {train.shape}   test {test.shape}")
train.head()

## 2 · Otomatik profil

Tek çağrı; elle 2 saat sürecek keşfin yerini alır. Özellikle şunları işaretler:
çarpıklık, sıfır yığılması, ID-benzeri kolonlar, yüksek kardinalite, şema farkı
(= sızıntı adayları) ve **birleşik nokta (U+0307)** taşıyan Türkçe kolonlar.

In [ ]:
# TODO: hedef kolon adını veri geldiğinde doldur
TARGET = "HEDEF_KOLON"

dataset_profile = profile(train, test, target=TARGET)
print(dataset_profile.report())

In [ ]:
# Kolon bazlı kompakt tablo — hızlı gözden geçirme için
quick_look(train)

## 3 · Doğrulama şeması — **yarışmanın kazanıldığı karar**

Yanlış şema iki şekilde öldürür:
- **İyimser CV:** lokal skor yüksek, leaderboard'da çakılıyorsun → sızıntı var
- **Gürültülü CV:** hangi değişikliğin işe yaradığını göremiyorsun → public LB'ye
  göre karar vermeye başlıyorsun → private LB'de çöküyorsun (shakeup)

In [ ]:
suggestion = suggest_scheme(train, target=TARGET)
print(suggestion)

## 4 · Sızıntı taraması

Modeli eğitmeden **önce** çalıştır. `critical` bulgular varsa dur ve çöz.

In [ ]:
TIME_COLUMN = None   # TODO: varsa zaman kolonu adı

findings = leakage_report(train, TARGET, test=test, time_column=TIME_COLUMN)
print(findings["summary"], "\n")

for severity in ("critical", "warning", "info"):
    for message in findings[severity]:
        print(f"[{severity.upper()}] {message}")

## 5 · Hedef dağılımı

Hedefin şekli metrik seçimini ve dönüşüm kararını belirler:
- **Çarpıklık > 2** → `log1p` dönüşümü dene
- **Sıfır yığılması > %40** → iki aşamalı model düşün (önce sıfır mı, sonra miktar)
- **Sınıf dengesizliği** → eşik optimizasyonu şart, 0.5 varsayılanı yanlış

In [ ]:
target_values = train[TARGET].dropna()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(target_values, bins=60, color="#4C6EF5", edgecolor="white", linewidth=0.4)
axes[0].set_title("Ham dağılım")
axes[0].set_xlabel(TARGET)

positive = target_values[target_values > 0]
axes[1].hist(np.log1p(positive), bins=60, color="#12B886", edgecolor="white", linewidth=0.4)
axes[1].set_title("log1p (yalnızca pozitifler)")

axes[2].boxplot(target_values, vert=True, widths=0.5)
axes[2].set_title("Kutu grafiği — aykırı değerler")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

print(f"çarpıklık = {target_values.skew():.3f}")
print(f"sıfır oranı = {(target_values == 0).mean():.3%}")
print(target_values.describe())

## 6 · Eksik veri haritası

Eksikliğin **rastgele olup olmadığı** önemlidir. Bir kolon yalnızca belirli bir
dönemde veya belirli bir varlıkta eksikse, bu bir sinyaldir — doldurmadan önce
`_eksikti` bayrağı ekle.

In [ ]:
missing = (train.isna().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0]

if len(missing):
    fig, ax = plt.subplots(figsize=(9, max(3, 0.32 * len(missing))))
    ax.barh(missing.index[::-1], missing.values[::-1], color="#FA5252")
    ax.set_xlabel("eksik %")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("Eksik değer yok.")

## 7 · Türkçe metin sağlığı

`İ` harfinin `.lower()` sonucu **iki kod noktasıdır** (U+0069 U+0307), dolayısıyla
`'İ'.lower() != 'i'`. İl/ilçe adıyla yapılan bir join **sessizce 0 satır** döner.
Harici veri (hava, nüfus) eklemeden önce bunu kontrol et.

In [ ]:
text_columns = categorical_columns(train)

for column in text_columns[:10]:
    sample = train[column].dropna().astype(str).head(300)
    if any(has_combining_dot(v) for v in sample):
        print(f"! {column}: BİRLEŞİK NOKTA var — yanlış .lower() kullanılmış")
    else:
        print(f"  {column}: temiz ({train[column].nunique()} benzersiz)")

# Kanıt: naif yaklaşım başarısız, join_key başarılı
print("\n'İ'.lower() =", codepoints("İ".lower()), "->", "İ".lower() == "i")
print("join_key('İZMİR') == join_key('Izmir') ->", join_key("İZMİR") == join_key("Izmir"))

## 8 · Zaman ekseni (varsa)

Train ve test'in zaman aralıkları ayrık mı? Ayrıksa **rastgele KFold geleceği
sızdırır** ve CV'yi yapay olarak yükseltir.

In [ ]:
if TIME_COLUMN:
    train_times = pd.to_datetime(train[TIME_COLUMN])
    test_times = pd.to_datetime(test[TIME_COLUMN])

    print(f"train: {train_times.min()} → {train_times.max()}")
    print(f"test:  {test_times.min()} → {test_times.max()}")
    print(f"boşluk: {test_times.min() - train_times.max()}")

    fig, ax = plt.subplots(figsize=(12, 3))
    ax.hist(train_times, bins=80, alpha=0.75, label="train", color="#4C6EF5")
    ax.hist(test_times, bins=40, alpha=0.75, label="test", color="#FA5252")
    ax.legend()
    ax.set_title("Zaman dağılımı")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

## 9 · Bulgular

> **Bu hücreyi doldur.** Jüri notebook'u okuyacak; burası "veriyi anladık"
> demenin yeri.

| # | Bulgu | Sonuç / aksiyon |
|---|-------|-----------------|
| 1 | | |
| 2 | | |
| 3 | | |

**Seçilen CV şeması:** …
**Tespit edilen sızıntı riskleri:** …
**İlk feature hipotezleri:** …